In [ ]:
import torch
import torchvision
import numpy as np
import torch.nn as nn
import torch.optim as optim
import time
from torchvision.transforms import v2
import os
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.drawing.image import Image as XLImage
import matplotlib.pyplot as plt
import io
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn, get_kl_loss

# Hyperparamètres
lr            = 0.005
epochs        = 50
batch_size    = 128
milestones    = [20, 35, 45]
gamma         = 0.5
num_monte_carlo = 25
N_RUNS        = 10

# Configurations : (nom, n_couches, moped, is_bayesian)
# n_couches=None : réseau complet sans split
CONFIGS = [
    ("BayMoped_n32_test",     32, True,  True),
]

# Fonctions utilitaires
def predictive_entropy(output):
    if output.ndim == 3:
        mean_output = np.mean(output, axis=0)
        return -np.sum(mean_output * np.log(mean_output + 1e-10), axis=-1)
    return -np.sum(output * np.log(output + 1e-10), axis=-1)

def mutual_information(output):
    if output.ndim == 3:
        mean_output = np.mean(output, axis=0)
        entropy = -np.sum(mean_output * np.log(mean_output + 1e-10), axis=-1)
        expected_entropy = -np.mean(np.sum(output * np.log(output + 1e-10), axis=-1), axis=0)
        return entropy - expected_entropy
    return np.zeros(output.shape[0], dtype=float)

def split_model_graph(model: nn.Module, n: int):
    """
    AI generated function.
    Splits a PyTorch model into two independently runnable models.
    Part B will contain the last N executed module calls.
    Part A will contain everything before that.
    """
    traced = torch.fx.symbolic_trace(model)
    nodes = list(traced.graph.nodes)
    
    # Find all actual module calls to determine the split point
    module_nodes = [node for node in nodes if node.op == 'call_module']
    if n >= len(module_nodes):
        raise ValueError(f"Cannot split off {n} layers; model only has {len(module_nodes)} module calls.")
        
    # The first node of Part B is the n-th module from the end
    first_node_of_b = module_nodes[-n]
    cut_idx = nodes.index(first_node_of_b)
    
    nodes_A = nodes[:cut_idx]
    nodes_B = nodes[cut_idx:]
    
    # 1. Identify "Boundary Nodes" 
    # (Tensors calculated in A that are needed in B)
    boundary_nodes = []
    for node in nodes_B:
        for in_node in node.all_input_nodes:
            if in_node in nodes_A and in_node not in boundary_nodes:
                boundary_nodes.append(in_node)
                
    # 2. Build Part A
    graph_A = torch.fx.Graph()
    env_A = {} # Maps old graph nodes to new Graph A nodes
    
    for node in nodes_A:
        env_A[node] = graph_A.node_copy(node, lambda x: env_A[x])
        
    # Set the outputs for Part A based on the boundary nodes we found
    output_args_A = tuple(env_A[node] for node in boundary_nodes)
    if len(output_args_A) == 1:
        graph_A.output(output_args_A[0])
    else:
        graph_A.output(output_args_A)
        
    model_A = torch.fx.GraphModule(traced, graph_A)
    
    # 3. Build Part B
    graph_B = torch.fx.Graph()
    env_B = {} # Maps old graph nodes to new Graph B nodes
    
    # Create input placeholders in B for the incoming tensors from A
    for node in boundary_nodes:
        env_B[node] = graph_B.placeholder(node.name)
        
    # Copy the remaining nodes into B
    for node in nodes_B:
        env_B[node] = graph_B.node_copy(node, lambda x: env_B[x])
        
    model_B = torch.fx.GraphModule(traced, graph_B)
    
    return model_A, model_B

from medmnist import PathMNIST
from medmnist import INFO

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

info = INFO['pathmnist']

trainset = PathMNIST(split="train", download=True, size=28, transform=transform)
valset   = PathMNIST(split="val",   download=True, size=28, transform=transform)
testset  = PathMNIST(split="test",  download=True, size=28, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
valloader   = torch.utils.data.DataLoader(valset,   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

class_names = info['label']
device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')

def build_model(n_couches, moped, is_bayesian):
    net = torchvision.models.efficientnet_b0(progress=True)
    net.classifier[1] = nn.Linear(1280, 9)

    if not is_bayesian:
        net.to(device)
        return net, None
    
    red = torch.load("effNet.pth", map_location=device)
    net.load_state_dict(red['model_state_dict'])

    const_bnn = {
        "prior_mu": 0.0, "prior_sigma": 1.0,
        "posterior_mu_init": 0.0, "posterior_rho_init": -3.0,
        "type": "Reparameterization",
        "moped_enable": moped, "moped_delta": 0.5,
    }

    if n_couches is None:
        dnn_to_bnn(net, const_bnn)
        net.to(device)
        return net, None
    else:
        model_A, model_B = split_model_graph(net, n=n_couches)
        dnn_to_bnn(model_B, const_bnn)
        model_A.to(device)
        model_B.to(device)
        model_A.eval()
        for param in model_A.parameters():
            param.requires_grad = False
        return model_B, model_A

def train_model(model, model_A, is_bayesian):
    criterion = nn.CrossEntropyLoss()
    params = model.parameters()
    optimizer = torch.optim.Adam(params, lr=lr)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=gamma)

    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        for data in trainloader:
            inputs, labels = data[0].to(device), data[1].to(device)
            labels = labels.squeeze(1)
            optimizer.zero_grad()

            if model_A is not None:
                with torch.no_grad():
                    inputs = model_A(inputs)

            output = model(inputs)

            if is_bayesian:
                kl = get_kl_loss(model)
                ce_loss = criterion(output, labels)
                loss = ce_loss + kl / len(trainset)
            else:
                loss = criterion(output, labels)

            loss.backward()
            optimizer.step()

        scheduler.step()

    elapsed = time.time() - start_time
    return elapsed

def evaluate_model(model, model_A, loader, is_bayesian, mc):
    model.eval()
    all_preds, all_labels, all_outputs = [], [], []

    with torch.no_grad():
        for data in loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            labels = labels.squeeze(1)

            output_mc = []
            for _ in range(mc):
                if model_A is not None:
                    feats = model_A(inputs)
                    logits = model(feats)
                else:
                    logits = model(inputs)
                probs = torch.nn.functional.softmax(logits, dim=-1)
                output_mc.append(probs)

            output = torch.stack(output_mc)
            pred_mean = output.mean(dim=0)
            y_pred = torch.argmax(pred_mean, dim=1)

            all_preds.append(y_pred.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_outputs.append(output.cpu().numpy())

    all_preds   = np.concatenate(all_preds,   axis=0)
    all_labels  = np.concatenate(all_labels,  axis=0)
    all_outputs = np.concatenate(all_outputs, axis=1)
    acc = (all_preds == all_labels).mean()
    return all_preds, all_labels, all_outputs, acc

def compute_metrics(all_preds, all_labels, all_outputs):
    # ACE
    n_bins = 15
    mean_probs = all_outputs.mean(axis=0)
    sorted_idx = np.argsort(mean_probs.max(axis=-1))
    conf_sorted = mean_probs.max(axis=-1)[sorted_idx]
    correct_sorted = (all_preds == all_labels).astype(float)[sorted_idx]
    bins = np.array_split(np.arange(len(all_labels)), n_bins)
    ace_bins_conf, ace_bins_acc, ace = [], [], 0
    for b in bins:
        if len(b) > 0:
            c = conf_sorted[b].mean()
            a = correct_sorted[b].mean()
            ace_bins_conf.append(c)
            ace_bins_acc.append(a)
            ace += len(b) * abs(a - c)
    ace /= len(all_labels)

    # AUCE
    predictive_uncertainty = predictive_entropy(all_outputs)
    errors = (all_preds != all_labels).astype(float)
    bin_boundaries = np.percentile(predictive_uncertainty, np.linspace(0, 100, n_bins + 1))
    bin_idx = np.digitize(predictive_uncertainty, bin_boundaries)
    auce_bin_unc, auce_bin_err, auce_bin_sizes = [], [], []
    for b in range(1, n_bins + 1):
        mask = bin_idx == b
        if mask.sum() > 0:
            auce_bin_unc.append(predictive_uncertainty[mask].mean())
            auce_bin_err.append(errors[mask].mean())
            auce_bin_sizes.append(mask.sum())
    unc_min, unc_max = predictive_uncertainty.min(), predictive_uncertainty.max()
    auce_unc_norm = [(u - unc_min) / (unc_max - unc_min + 1e-10) for u in auce_bin_unc]
    auce = sum(s * abs(e - u) for s, e, u in zip(auce_bin_sizes, auce_bin_err, auce_unc_norm)) / len(all_labels)

    return ace, auce, ace_bins_conf, ace_bins_acc, auce_bin_unc, auce_bin_err

# Excel
def make_chart_buf(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=120)
    plt.close(fig)
    buf.seek(0)
    return buf

def save_config_results(config_name, results, filename="res.xlsx"):
    header_font = Font(bold=True, color="FFFFFF", name="Arial")
    header_fill = PatternFill("solid", start_color="2F4F8F")
    normal_font = Font(name="Arial")
    mean_fill   = PatternFill("solid", start_color="E8F4FD")
    std_fill    = PatternFill("solid", start_color="FDF3E8")

    def set_header(cell, value):
        cell.value = value; cell.font = header_font; cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center")

    def set_col_width(ws, col, width):
        ws.column_dimensions[openpyxl.utils.get_column_letter(col)].width = width

    wb = openpyxl.load_workbook(filename) if os.path.exists(filename) else openpyxl.Workbook()
    if wb.active.title == "Sheet": wb.active.title = "Summary"

    # Feuille Summary
    if "Summary" not in wb.sheetnames:
        ws_sum = wb.create_sheet("Summary", 0)
    else:
        ws_sum = wb["Summary"]

    sum_headers = ["Config", "Mean Train Acc (%)", "Std Train Acc",
                   "Mean Val Acc (%)", "Std Val Acc",
                   "Mean Test Acc (%)", "Std Test Acc",
                   "Mean ACE (%)", "Std ACE",
                   "Mean AUCE (%)", "Std AUCE",
                   "Mean Time (s)", "Std Time"]

    if ws_sum.cell(1, 1).value is None:
        for col, h in enumerate(sum_headers, start=1):
            set_header(ws_sum.cell(1, col), h)
            set_col_width(ws_sum, col, 20)

    accs_train = [r['train_acc'] for r in results]
    accs_val   = [r['val_acc']   for r in results]
    accs_test  = [r['test_acc']  for r in results]
    aces       = [r['ace']       for r in results]
    auces      = [r['auce']      for r in results]
    times      = [r['time']      for r in results]

    row = ws_sum.max_row + 1
    sum_values = [
        config_name,
        round(100 * np.mean(accs_train), 2), round(100 * np.std(accs_train), 2),
        round(100 * np.mean(accs_val),   2), round(100 * np.std(accs_val),   2),
        round(100 * np.mean(accs_test),  2), round(100 * np.std(accs_test),  2),
        round(100 * np.mean(aces),       2), round(100 * np.std(aces),       2),
        round(100 * np.mean(auces),      2), round(100 * np.std(auces),      2),
        round(np.mean(times), 1),             round(np.std(times), 1),
    ]
    for col, val in enumerate(sum_values, start=1):
        cell = ws_sum.cell(row, col, val)
        cell.font = normal_font

    # Feuille détail config
    ws2 = wb.create_sheet(config_name[:30])

    detail_headers = ["Run", "Train Acc (%)", "Val Acc (%)", "Test Acc (%)",
                      "ACE (%)", "AUCE (%)", "Time (s)"]
    for col, h in enumerate(detail_headers, start=1):
        set_header(ws2.cell(1, col), h)
        set_col_width(ws2, col, 16)

    for i, r in enumerate(results, start=2):
        vals = [i - 1,
                round(100 * r['train_acc'], 2), round(100 * r['val_acc'], 2),
                round(100 * r['test_acc'], 2),  round(100 * r['ace'], 2),
                round(100 * r['auce'], 2),       round(r['time'], 1)]
        for col, val in enumerate(vals, start=1):
            ws2.cell(i, col, val).font = normal_font

    # Ligne moyenne
    mean_row = len(results) + 2
    ws2.cell(mean_row, 1, "Mean").font = Font(bold=True, name="Arial")
    mean_vals = [round(100 * np.mean(accs_train), 2), round(100 * np.mean(accs_val), 2),
                 round(100 * np.mean(accs_test),  2), round(100 * np.mean(aces), 2),
                 round(100 * np.mean(auces), 2),       round(np.mean(times), 1)]
    for col, val in enumerate(mean_vals, start=2):
        c = ws2.cell(mean_row, col, val)
        c.font = Font(bold=True, name="Arial"); c.fill = mean_fill

    # Ligne écart type
    std_row = mean_row + 1
    ws2.cell(std_row, 1, "Std").font = Font(bold=True, name="Arial")
    std_vals = [round(100 * np.std(accs_train), 2), round(100 * np.std(accs_val), 2),
                round(100 * np.std(accs_test),  2), round(100 * np.std(aces), 2),
                round(100 * np.std(auces), 2),       round(np.std(times), 1)]
    for col, val in enumerate(std_vals, start=2):
        c = ws2.cell(std_row, col, val)
        c.font = Font(bold=True, name="Arial"); c.fill = std_fill

    # Grille commune ACE
    conf_grid = np.linspace(0, 1, 50)
    ace_acc_all = [np.interp(conf_grid, r['ace_bins_conf'], r['ace_bins_acc']) for r in results]
    mean_ace_acc = np.mean(ace_acc_all, axis=0)
    std_ace_acc  = np.std(ace_acc_all,  axis=0)

    # Grille commune AUCE
    unc_grid = np.linspace(0, max(r['auce_bin_unc'][-1] for r in results), 50)
    auce_err_all = [np.interp(unc_grid, r['auce_bin_unc'], r['auce_bin_err']) for r in results]
    mean_auce_err = np.mean(auce_err_all, axis=0)
    std_auce_err  = np.std(auce_err_all,  axis=0)

    set_header(ws2.cell(1, 10), "ACE - Confidence")
    set_header(ws2.cell(1, 11), "ACE - Accuracy (mean)")
    set_header(ws2.cell(1, 12), "ACE - Accuracy (std)")
    set_header(ws2.cell(1, 14), "AUCE - Uncertainty")
    set_header(ws2.cell(1, 15), "AUCE - Error rate (mean)")
    set_header(ws2.cell(1, 16), "AUCE - Error rate (std)")
    for col in [10,11,12,14,15,16]: set_col_width(ws2, col, 22)

    for i, (c, a, s) in enumerate(zip(conf_grid, mean_ace_acc, std_ace_acc), start=2):
        ws2.cell(i, 10, round(float(c), 4)).font = normal_font
        ws2.cell(i, 11, round(float(a), 4)).font = normal_font
        ws2.cell(i, 12, round(float(s), 4)).font = normal_font

    # Graphique ACE
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(conf_grid, mean_ace_acc, color='steelblue', marker='o', markersize=3, label="Mean")
    ax.fill_between(conf_grid, mean_ace_acc - std_ace_acc,
                                mean_ace_acc + std_ace_acc, alpha=0.2, color='steelblue', label="±1 std")
    ax.plot([0,1],[0,1],'--',color='gray',label="Perfect")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_title(f"{config_name} — ACE ({round(100*np.mean(aces),2)}%)")
    ax.legend(); ax.grid(True, alpha=0.3)
    img = XLImage(make_chart_buf(fig)); img.anchor = "P1"; ws2.add_image(img)

    # Graphique AUCE
    unc_grid_norm = (unc_grid - unc_grid.min()) / (unc_grid.max() - unc_grid.min())

    for i, (u, e, s) in enumerate(zip(unc_grid_norm, mean_auce_err, std_auce_err), start=2):
        ws2.cell(i, 14, round(float(u), 4)).font = normal_font
        ws2.cell(i, 15, round(float(e), 4)).font = normal_font
        ws2.cell(i, 16, round(float(s), 4)).font = normal_font

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(unc_grid_norm, mean_auce_err, color='steelblue', marker='o', markersize=3, label="Mean")
    ax.fill_between(unc_grid_norm, mean_auce_err - std_auce_err,
                                mean_auce_err + std_auce_err, alpha=0.2, color='steelblue', label="±1 std")
    ax.set_xlabel("Normalized Uncertainty"); ax.set_ylabel("Error Rate")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_title(f"{config_name} — AUCE ({round(100*np.mean(auces),2)}%)")
    ax.legend(); ax.grid(True, alpha=0.3)
    img = XLImage(make_chart_buf(fig)); img.anchor = "P25"; ws2.add_image(img)

    wb.save(filename)
    print(f"Config {config_name} sauvegardée dans {filename}")


# Boucle principale
def run_all_configs():
    mc_map = {"EffNetBase": 1}  # mc=1 pour le modèle non bayésien

    for config_name, n_couches, moped, is_bayesian in CONFIGS:
        mc = mc_map.get(config_name, num_monte_carlo)
        print(f"\n{'='*50}")
        print(f"Config : {config_name}  ({N_RUNS} runs)")
        print(f"{'='*50}")

        results = []

        for run_i in range(N_RUNS):
            print(f"  Run {run_i + 1}/{N_RUNS}...")
            save_path = f"{config_name}_run{run_i}.pth"

            # Entraînement si pas déjà sauvegardé
            if not os.path.exists(save_path):
                model, model_A = build_model(n_couches, moped, is_bayesian)
                elapsed = train_model(model, model_A, is_bayesian)

                torch.save({
                    'model_state_dict': model.state_dict(),
                    'model_A_state_dict': model_A.state_dict() if model_A is not None else None,
                    'time': elapsed,
                    'n_couches': n_couches,
                    'moped': moped,
                    'is_bayesian': is_bayesian,
                }, save_path)
                print(f"    Entraîné en {elapsed:.1f}s → {save_path}")
            else:
                print(f"    Déjà entraîné, chargement de {save_path}")

            # Chargement et évaluation
            checkpoint = torch.load(save_path, map_location=device)
            elapsed = checkpoint['time']

            model, model_A = build_model(n_couches, moped, is_bayesian)
            model.load_state_dict(checkpoint['model_state_dict'])
            if model_A is not None and checkpoint['model_A_state_dict'] is not None:
                model_A.load_state_dict(checkpoint['model_A_state_dict'])

            all_preds, all_labels, all_outputs, test_acc = evaluate_model(model, model_A, testloader,  is_bayesian, mc)
            _, _, _, train_acc = evaluate_model(model, model_A, trainloader, is_bayesian, mc)
            _, _, _, val_acc   = evaluate_model(model, model_A, valloader,   is_bayesian, mc)

            ace, auce, ace_bins_conf, ace_bins_acc, auce_bin_unc, auce_bin_err = compute_metrics(all_preds, all_labels, all_outputs)

            results.append({
                'train_acc': train_acc, 'val_acc': val_acc, 'test_acc': test_acc,
                'ace': ace, 'auce': auce, 'time': elapsed,
                'ace_bins_conf': ace_bins_conf, 'ace_bins_acc': ace_bins_acc,
                'auce_bin_unc': auce_bin_unc, 'auce_bin_err': auce_bin_err,
            })

            print(f"    Test acc: {100*test_acc:.2f}% | ACE: {100*ace:.2f}% | AUCE: {100*auce:.2f}%")

        save_config_results(config_name, results, filename="res.xlsx")
        print(f"  Moyenne test acc : {100*np.mean([r['test_acc'] for r in results]):.2f}% "
              f"± {100*np.std([r['test_acc'] for r in results]):.2f}%")

# Lancer
run_all_configs() 


Config : BayMoped_n32_test  (10 runs)
  Run 1/10...
    Entraîné en 786.8s → BayMoped_n32_test_run0.pth
    Test acc: 84.04% | ACE: 10.81% | AUCE: 7.80%
  Run 2/10...
    Entraîné en 791.7s → BayMoped_n32_test_run1.pth
    Test acc: 82.59% | ACE: 11.01% | AUCE: 7.53%
  Run 3/10...
    Entraîné en 788.4s → BayMoped_n32_test_run2.pth
    Test acc: 82.41% | ACE: 6.78% | AUCE: 6.46%
  Run 4/10...
    Entraîné en 789.3s → BayMoped_n32_test_run3.pth
    Test acc: 81.73% | ACE: 8.56% | AUCE: 4.67%
  Run 5/10...
    Entraîné en 793.6s → BayMoped_n32_test_run4.pth
    Test acc: 71.20% | ACE: 10.77% | AUCE: 10.53%
  Run 6/10...
    Entraîné en 797.0s → BayMoped_n32_test_run5.pth
    Test acc: 77.70% | ACE: 10.81% | AUCE: 7.20%
  Run 7/10...
    Entraîné en 790.6s → BayMoped_n32_test_run6.pth
    Test acc: 81.00% | ACE: 12.74% | AUCE: 9.10%
  Run 8/10...
    Entraîné en 792.4s → BayMoped_n32_test_run7.pth
    Test acc: 82.49% | ACE: 12.22% | AUCE: 9.91%
  Run 9/10...
    Entraîné en 796.4s → Bay